# KADIMA Mini Job Matcher

## Hackathon 2 — AI-Powered Document Search and Summarization System

### Project Overview

KADIMA Mini Job Matcher is an AI-powered tool designed to help job seekers identify career opportunities that match their professional experience and skills.

### Problem

Job seekers often spend significant time reviewing job descriptions to determine whether their qualifications match the requirements.

### Solution

KADIMA uses semantic search and AI-powered summarization to compare a candidate's resume with job descriptions and identify relevant opportunities.

### Key Features

- Resume processing (PDF or TXT)
- Job description analysis
- Semantic search using Sentence Transformers
- Vector similarity search using FAISS
- AI-powered job summarization
- Top job recommendations based on semantic relevance

### Technologies

Python | Hugging Face | Sentence Transformers | FAISS | Google Colab

### Business Value

KADIMA aims to reduce the time spent manually reviewing job opportunities and support more informed career decisions.

In [1]:
# Install required libraries for the KADIMA Mini Job Matcher
%pip install -q sentence-transformers faiss-cpu transformers pypdf python-docx pandas

In [2]:
# Verify required libraries
import sentence_transformers
import faiss
import transformers
import pypdf
import docx
import pandas

print("All required libraries imported successfully!")

All required libraries imported successfully!


In [11]:
import pandas as pd

# Create a sample dataset with detailed job descriptions
jobs = [
    {
        "job_id": 1,
        "title": "Enterprise Account Executive",
        "description": """
        The Enterprise Account Executive is responsible for managing
        strategic enterprise customers and developing new business
        opportunities across the technology sector. The role involves
        selling cloud infrastructure, SaaS platforms, and enterprise
        software solutions to large organizations. Responsibilities
        include prospecting, qualifying leads, conducting discovery
        meetings, developing sales pipelines, delivering commercial
        presentations, and negotiating complex contracts. The candidate
        must collaborate with technical teams, understand customer
        business requirements, manage CRM records, and maintain accurate
        revenue forecasts. Strong communication, relationship management,
        consultative selling, and negotiation skills are essential.
        Experience in B2B technology sales and enterprise account
        management is highly desirable.
        """
    },
    {
        "job_id": 2,
        "title": "Customer Success Manager",
        "description": """
        The Customer Success Manager manages relationships with existing
        customers to ensure successful product adoption and long-term
        satisfaction. Responsibilities include customer onboarding,
        conducting regular business reviews, monitoring account health,
        identifying customer needs, and developing retention strategies.
        The professional works closely with sales, support, and product
        teams to resolve issues and improve the customer experience.
        Additional responsibilities include identifying upselling and
        cross-selling opportunities, reducing customer churn, tracking
        customer satisfaction metrics, and managing renewal processes.
        Candidates should have experience with SaaS platforms, CRM
        systems, customer relationship management, and business
        communication. Strong analytical, problem-solving, and
        interpersonal skills are required.
        """
    },
    {
        "job_id": 3,
        "title": "Python Backend Developer",
        "description": """
        The Python Backend Developer designs, develops, and maintains
        server-side applications supporting modern web platforms.
        Responsibilities include building REST APIs, implementing
        business logic, integrating external services, and managing
        relational databases such as PostgreSQL. The developer
        collaborates with frontend engineers, product managers, and
        quality assurance teams to deliver reliable software.
        Additional tasks include writing automated tests, debugging
        applications, optimizing performance, reviewing code, and
        maintaining technical documentation. Experience with Python,
        FastAPI or Django, SQL, Git, Docker, and cloud infrastructure
        is desirable. Candidates should understand software
        architecture, security practices, version control, and
        agile development methodologies.
        """
    },
    {
        "job_id": 4,
        "title": "Sales Operations Analyst",
        "description": """
        The Sales Operations Analyst supports commercial teams by
        improving sales processes, reporting, and operational
        efficiency. Responsibilities include maintaining CRM data,
        analyzing pipeline performance, monitoring revenue forecasts,
        and developing dashboards for sales management. The analyst
        identifies process bottlenecks, evaluates conversion rates,
        supports territory planning, and prepares business reports.
        Collaboration with sales leadership, finance, and marketing
        teams is essential to ensure consistent performance metrics.
        Candidates should have strong analytical skills and experience
        with Excel, CRM platforms, reporting tools, and business
        intelligence dashboards. Knowledge of Salesforce, SQL,
        sales forecasting, and process automation is advantageous.
        """
    },
    {
        "job_id": 5,
        "title": "Data Scientist",
        "description": """
        The Data Scientist analyzes structured and unstructured
        datasets to identify patterns, generate insights, and support
        business decisions. Responsibilities include data cleaning,
        exploratory analysis, feature engineering, statistical
        modeling, and machine learning development. The professional
        collaborates with business stakeholders to define analytical
        problems and translate findings into actionable recommendations.
        Additional responsibilities include evaluating model
        performance, creating visualizations, documenting experiments,
        and presenting results to technical and nontechnical audiences.
        Candidates should have experience with Python, pandas,
        NumPy, scikit-learn, SQL, and statistical methods.
        Familiarity with predictive analytics, cloud platforms,
        and data visualization tools is beneficial.
        """
    },
    {
        "job_id": 6,
        "title": "Business Development Manager",
        "description": """
        The Business Development Manager identifies new commercial
        opportunities and develops strategies to expand the company's
        market presence. Responsibilities include researching target
        markets, prospecting potential customers, establishing
        strategic partnerships, and negotiating business agreements.
        The professional manages relationships with prospective
        clients, develops go-to-market strategies, and collaborates
        with sales and marketing teams to generate qualified leads.
        Additional responsibilities include evaluating competitors,
        identifying emerging market trends, maintaining sales
        forecasts, and presenting growth opportunities to management.
        Candidates should demonstrate strong communication,
        negotiation, relationship-building, and strategic planning
        skills. Experience in B2B sales, technology solutions,
        and international business development is advantageous.
        """
    },
    {
        "job_id": 7,
        "title": "Cybersecurity Engineer",
        "description": """
        The Cybersecurity Engineer protects enterprise systems,
        applications, and networks against security threats.
        Responsibilities include implementing security controls,
        monitoring vulnerabilities, analyzing security incidents,
        and supporting incident response activities. The engineer
        evaluates infrastructure configurations, conducts security
        assessments, and collaborates with IT teams to reduce
        organizational risks. Additional tasks include maintaining
        security documentation, reviewing access permissions,
        supporting compliance requirements, and recommending
        improvements to cloud and network security.
        Candidates should understand network protocols, identity
        management, encryption, firewalls, vulnerability scanning,
        and security monitoring tools. Experience with cloud
        security, Python scripting, and security frameworks
        is desirable.
        """
    },
    {
        "job_id": 8,
        "title": "Technical Project Manager",
        "description": """
        The Technical Project Manager leads technology initiatives
        from planning through implementation and delivery.
        Responsibilities include defining project scope, developing
        schedules, managing budgets, coordinating cross-functional
        teams, and monitoring project risks. The professional
        communicates with stakeholders, tracks milestones, resolves
        delivery issues, and ensures that technical solutions meet
        business requirements. Additional responsibilities include
        managing vendors, preparing project reports, facilitating
        meetings, and maintaining project documentation.
        Candidates should have experience with software
        implementation, systems integration, agile methodologies,
        and project management tools. Strong leadership,
        communication, organization, and problem-solving skills
        are required. Knowledge of Scrum, Jira, and PMP
        methodologies is beneficial.
        """
    },
    {
        "job_id": 9,
        "title": "UX/UI Designer",
        "description": """
        The UX/UI Designer creates intuitive and visually consistent
        digital experiences for web and mobile applications.
        Responsibilities include researching user needs, designing
        wireframes, building interactive prototypes, and developing
        interface layouts. The designer collaborates with product
        managers, developers, and stakeholders to translate business
        requirements into user-friendly solutions. Additional tasks
        include conducting usability testing, analyzing user
        feedback, maintaining design systems, and improving
        accessibility across digital products. Candidates should
        understand user-centered design principles, information
        architecture, responsive design, and visual communication.
        Experience with Figma, prototyping tools, and collaborative
        product development is desirable.
        """
    },
    {
        "job_id": 10,
        "title": "Solutions Consultant",
        "description": """
        The Solutions Consultant supports enterprise sales by
        translating customer business requirements into appropriate
        technology solutions. Responsibilities include conducting
        discovery sessions, delivering technical demonstrations,
        preparing proof-of-concept presentations, and explaining
        software capabilities to prospective customers.
        The consultant collaborates with account executives,
        product specialists, and engineering teams throughout
        the sales cycle. Additional responsibilities include
        responding to technical questions, supporting proposals,
        documenting solution requirements, and identifying
        integration opportunities. Candidates should possess
        strong communication, analytical, and presentation skills.
        Experience with enterprise software, SaaS platforms,
        APIs, cloud technologies, and consultative selling
        is highly desirable.
        """
    }
]

jobs_df = pd.DataFrame(jobs)
jobs_df["description"] = (
    jobs_df["description"]
    .str.split()
    .str.join(" ")
)

jobs_df.to_csv("sample_jobs.csv", index=False)

print(f"Dataset created successfully: {len(jobs_df)} jobs")
print("Average description length:",
      round(jobs_df["description"].str.split().str.len().mean()),
      "words")

display(jobs_df)

jobs_df = pd.DataFrame(jobs)
jobs_df.to_csv("sample_jobs.csv", index=False)

print(f"Dataset created successfully: {len(jobs_df)} jobs")
display(jobs_df)

Dataset created successfully: 10 jobs
Average description length: 92 words


,job_id,title,description
0,1,Enterprise Account Executive,The Enterprise Account Executive is responsibl...
1,2,Customer Success Manager,The Customer Success Manager manages relations...
2,3,Python Backend Developer,"The Python Backend Developer designs, develops..."
3,4,Sales Operations Analyst,The Sales Operations Analyst supports commerci...
4,5,Data Scientist,The Data Scientist analyzes structured and uns...
5,6,Business Development Manager,The Business Development Manager identifies ne...
6,7,Cybersecurity Engineer,The Cybersecurity Engineer protects enterprise...
7,8,Technical Project Manager,The Technical Project Manager leads technology...
8,9,UX/UI Designer,The UX/UI Designer creates intuitive and visua...
9,10,Solutions Consultant,The Solutions Consultant supports enterprise s...


Dataset created successfully: 10 jobs


,job_id,title,description
0,1,Enterprise Account Executive,\n The Enterprise Account Executive is ...
1,2,Customer Success Manager,\n The Customer Success Manager manages...
2,3,Python Backend Developer,\n The Python Backend Developer designs...
3,4,Sales Operations Analyst,\n The Sales Operations Analyst support...
4,5,Data Scientist,\n The Data Scientist analyzes structur...
5,6,Business Development Manager,\n The Business Development Manager ide...
6,7,Cybersecurity Engineer,\n The Cybersecurity Engineer protects ...
7,8,Technical Project Manager,\n The Technical Project Manager leads ...
8,9,UX/UI Designer,\n The UX/UI Designer creates intuitive...
9,10,Solutions Consultant,\n The Solutions Consultant supports en...


In [12]:
from sentence_transformers import SentenceTransformer

# Load the sentence embedding model
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

# Generate embeddings for job titles and descriptions
job_texts = (
    jobs_df["title"] + ". " + jobs_df["description"]
).tolist()

job_embeddings = embedding_model.encode(
    job_texts,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embeddings generated successfully!")
print("Number of jobs:", len(job_embeddings))
print("Embedding dimensions:", job_embeddings.shape[1])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embeddings generated successfully!
Number of jobs: 10
Embedding dimensions: 384


In [13]:
import faiss
import numpy as np

# Prepare embeddings for FAISS
embeddings_array = np.asarray(job_embeddings, dtype="float32")

# Create a FAISS index using inner product similarity
dimension = embeddings_array.shape[1]
faiss_index = faiss.IndexFlatIP(dimension)

# Add job embeddings to the index
faiss_index.add(embeddings_array)

print("FAISS index created successfully!")
print("Total indexed jobs:", faiss_index.ntotal)
print("Vector dimensions:", faiss_index.d)

FAISS index created successfully!
Total indexed jobs: 10
Vector dimensions: 384


In [6]:
# Search for jobs using semantic similarity
def search_jobs(candidate_profile, top_k=3):
    query_embedding = embedding_model.encode(
        [candidate_profile],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = faiss_index.search(query_embedding, top_k)

    results = []

    for score, index in zip(scores[0], indices[0]):
        job = jobs_df.iloc[int(index)]

        results.append({
            "job_title": job["title"],
            "description": job["description"],
            "similarity_score": round(float(score), 4)
        })

    return pd.DataFrame(results)


print("Semantic job search function created successfully!")

Semantic job search function created successfully!


In [7]:
# Test semantic job matching with a sample candidate profile
candidate_profile = """
Experienced technology sales professional with expertise in
enterprise account management, SaaS solutions, business development,
strategic partnerships, customer success, and sales negotiations.
"""

matching_jobs = search_jobs(candidate_profile, top_k=3)

print("Top 3 Recommended Jobs:")
display(matching_jobs)

Top 3 Recommended Jobs:


,job_title,description,similarity_score
0,Solutions Consultant,Support enterprise sales through technical dem...,0.6821
1,Enterprise Account Executive,"Manage enterprise customers, develop sales pip...",0.6235
2,Business Development Manager,"Identify business opportunities, build strateg...",0.5610


In [17]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load a pretrained model specialized in text summarization
summary_model_name = "facebook/bart-large-cnn"

summary_tokenizer = AutoTokenizer.from_pretrained(summary_model_name)

summary_model = AutoModelForSeq2SeqLM.from_pretrained(
    summary_model_name
)

print("BART summarization model loaded successfully!")

config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

BART summarization model loaded successfully!


In [22]:
# Generate a concise summary using BART
def summarize_job(job_description):
    inputs = summary_tokenizer(
        job_description,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    outputs = summary_model.generate(
    **inputs,
    max_length=100,
    min_length=25,
    num_beams=4,
    no_repeat_ngram_size=3,
    do_sample=False
)

    return summary_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )


print("BART summarization function updated successfully!")

BART summarization function updated successfully!


In [23]:
# Test summarization with an expanded job description
job = jobs_df[jobs_df["title"] == "Solutions Consultant"].iloc[0]

job_title = job["title"]
job_description = job["description"]
job_summary = summarize_job(job_description)

print("Job Title:", job_title)
print("\nOriginal Description:", job_description)
print("\nOriginal Word Count:", len(job_description.split()))
print("\nAI-Generated Summary:", job_summary)
print("Summary Word Count:", len(job_summary.split()))

Job Title: Solutions Consultant

Original Description: 
        The Solutions Consultant supports enterprise sales by
        translating customer business requirements into appropriate
        technology solutions. Responsibilities include conducting
        discovery sessions, delivering technical demonstrations,
        preparing proof-of-concept presentations, and explaining
        software capabilities to prospective customers.
        The consultant collaborates with account executives,
        product specialists, and engineering teams throughout
        the sales cycle. Additional responsibilities include
        responding to technical questions, supporting proposals,
        documenting solution requirements, and identifying
        integration opportunities. Candidates should possess
        strong communication, analytical, and presentation skills.
        Experience with enterprise software, SaaS platforms,
        APIs, cloud technologies, and consultative selling
      

In [20]:
# Combine semantic search and AI-powered summarization
def recommend_jobs(candidate_profile, top_k=3):
    matching_jobs = search_jobs(candidate_profile, top_k)

    recommendations = []

    for _, job in matching_jobs.iterrows():
        summary = summarize_job(job["description"])

        recommendations.append({
            "Job Title": job["job_title"],
            "Similarity Score": job["similarity_score"],
            "AI Summary": summary
        })

    return pd.DataFrame(recommendations)


print("KADIMA recommendation engine created successfully!")

KADIMA recommendation engine created successfully!


In [24]:
# Test the complete KADIMA recommendation workflow
candidate_profile = """
Experienced technology sales professional with expertise in
enterprise account management, SaaS solutions, business development,
strategic partnerships, customer success, and sales negotiations.
"""

recommendations = recommend_jobs(
    candidate_profile,
    top_k=3
)

print("KADIMA Mini Job Matcher - Top Recommendations")
display(recommendations)

KADIMA Mini Job Matcher - Top Recommendations


,Job Title,Similarity Score,AI Summary
0,Solutions Consultant,0.6771,The Solutions Consultant supports enterprise s...
1,Enterprise Account Executive,0.5980,The Enterprise Account Executive is responsibl...
2,Business Development Manager,0.5570,Business Development Manager identifies new co...


In [25]:
# Verify Gradio availability
import gradio as gr

print("Gradio version:", gr.__version__)
print("Gradio imported successfully!")

Gradio version: 6.30.0
Gradio imported successfully!


In [33]:
# KADIMA Mini Job Matcher - Gradio Interface

import gradio as gr

def match_candidate(profile, resume_file):
    # Extract text from an uploaded resume, if provided
    if resume_file is not None:
        profile = extract_resume_text(resume_file)

    # Validate candidate input
    if not profile or not profile.strip():
        raise gr.Error(
            "Please enter a professional profile or upload a resume."
        )

    # Limit input length for this prototype
    profile = profile[:1500]

    # Generate the top 3 job recommendations
    return recommend_jobs(profile, top_k=3)


with gr.Blocks(title="KADIMA Mini Job Matcher") as demo:

    gr.Markdown("""
    # KADIMA Mini Job Matcher
    ### AI-Powered Job Search and Summarization

    Discover relevant job opportunities using semantic search
    and artificial intelligence.

    Enter your professional profile or upload your resume.
    """)

    profile_input = gr.Textbox(
        label="Professional Profile",
        placeholder="Describe your experience, skills, and career interests...",
        lines=5
    )

    resume_input = gr.File(
        label="Upload Resume (PDF or TXT)",
        file_types=[".pdf", ".txt"],
        type="filepath"
    )

    search_button = gr.Button(
        "Find Matching Jobs",
        variant="primary"
    )

    results_output = gr.Dataframe(
        label="Top 3 Job Recommendations",
        interactive=False
    )

    search_button.click(
        fn=match_candidate,
        inputs=[profile_input, resume_input],
        outputs=results_output
    )


# Launch the application in Google Colab
demo.launch(share=False, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

In [27]:
# Evaluate semantic search using Precision@3

test_cases = [
    {
        "profile": "Enterprise software sales, account management, SaaS and business development.",
        "relevant_jobs": [
            "Enterprise Account Executive",
            "Business Development Manager",
            "Solutions Consultant"
        ]
    },
    {
        "profile": "Python programming, backend development, REST APIs and SQL databases.",
        "relevant_jobs": [
            "Python Backend Developer"
        ]
    },
    {
        "profile": "Cybersecurity, network protection, vulnerability assessment and cloud security.",
        "relevant_jobs": [
            "Cybersecurity Engineer"
        ]
    }
]

precision_scores = []

for i, test in enumerate(test_cases, start=1):
    results = search_jobs(test["profile"], top_k=3)

    predicted_jobs = results["job_title"].tolist()

    relevant_found = len(
        set(predicted_jobs) & set(test["relevant_jobs"])
    )

    precision = relevant_found / 3
    precision_scores.append(precision)

    print(f"Test {i}: {predicted_jobs}")
    print(f"Precision@3: {precision:.2f}\n")

print(f"Mean Precision@3: {sum(precision_scores) / len(precision_scores):.2f}")

Test 1: ['Enterprise Account Executive', 'Solutions Consultant', 'Business Development Manager']
Precision@3: 1.00

Test 2: ['Python Backend Developer', 'Data Scientist', 'Cybersecurity Engineer']
Precision@3: 0.33

Test 3: ['Cybersecurity Engineer', 'Python Backend Developer', 'Data Scientist']
Precision@3: 0.33

Mean Precision@3: 0.56


In [28]:
# Evaluate summarization compression

summary_results = []

for _, job in recommendations.iterrows():
    original = jobs_df.loc[
        jobs_df["title"] == job["Job Title"],
        "description"
    ].iloc[0]

    summary = job["AI Summary"]

    original_words = len(original.split())
    summary_words = len(summary.split())

    reduction = (1 - summary_words / original_words) * 100

    summary_results.append({
        "Job Title": job["Job Title"],
        "Original Words": original_words,
        "Summary Words": summary_words,
        "Reduction (%)": round(reduction, 1)
    })

summary_evaluation = pd.DataFrame(summary_results)

display(summary_evaluation)

print(
    "Average Word Reduction:",
    round(summary_evaluation["Reduction (%)"].mean(), 1),
    "%"
)

,Job Title,Original Words,Summary Words,Reduction (%)
0,Solutions Consultant,88,39,55.7
1,Enterprise Account Executive,95,31,67.4
2,Business Development Manager,94,31,67.0


Average Word Reduction: 63.4 %


In [29]:
# Evaluate summarization with ROUGE-1

from collections import Counter
import re

def rouge1_f1(reference, generated):
    def tokenize(text):
        return re.findall(r"\b\w+\b", text.lower())

    reference_tokens = Counter(tokenize(reference))
    generated_tokens = Counter(tokenize(generated))

    overlap = sum((reference_tokens & generated_tokens).values())

    precision = overlap / sum(generated_tokens.values()) if generated_tokens else 0
    recall = overlap / sum(reference_tokens.values()) if reference_tokens else 0

    return 2 * precision * recall / (precision + recall) if precision + recall else 0


reference_summary = (
    "The Solutions Consultant supports enterprise sales by understanding "
    "customer requirements, presenting software solutions, conducting "
    "demonstrations, and communicating technical capabilities."
)

generated_summary = recommendations.loc[
    recommendations["Job Title"] == "Solutions Consultant",
    "AI Summary"
].iloc[0]

score = rouge1_f1(reference_summary, generated_summary)

print(f"ROUGE-1 F1 Score: {score:.3f}")

ROUGE-1 F1 Score: 0.552


In [30]:
# Extract text from PDF and TXT resume files

from pypdf import PdfReader

def extract_resume_text(file_path):
    if not file_path:
        return ""

    file_path = str(file_path)

    if file_path.lower().endswith(".pdf"):
        reader = PdfReader(file_path)

        text = "\n".join(
            page.extract_text() or ""
            for page in reader.pages
        )

    elif file_path.lower().endswith(".txt"):
        with open(file_path, "r", encoding="utf-8-sig") as file:
            text = file.read()

    else:
        raise ValueError("Unsupported file format. Please upload PDF or TXT.")

    return " ".join(text.split())


print("PDF and TXT resume extraction function created successfully!")

PDF and TXT resume extraction function created successfully!
